# # Load training data

In [ ]:
import pandas as pd
from nubison_model import data

raw = pd.DataFrame({
    "text": [
        "I love this product", "Amazing experience", "Wonderful day",
        "Best ever made", "Excellent quality", "Pretty good overall",
        "Terrible service", "I hate it", "Awful experience",
        "Worst day ever", "Disappointing result", "Not good at all",
    ],
    "target": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})

datasets = data.split(raw, {"train": 0.75, "val": 0.25}, random_state=42)
for name, sub in datasets.items():
    print(f"{name:6s} rows={len(sub):3d}")


# # Train (HuggingFace transformers)

In [ ]:
# `train(datasets, target, *, ...)` parameters:
#   datasets      — dict from `data.split` (must include "train";
#                   "val" enables `t.X_val / t.y_val` + auto-scoring;
#                   "test" populates `t.X_test / t.y_test`)
#   target        — label column name(s); single string or list of strings.
#                   Splits each split's DataFrame into X = features and
#                   y = target so estimators can call `fit(X, y)`.
#   model_type    — free-form string tagged on the run as `model_type`
#                   (surfaced in the nubison UI). "classifier" and
#                   "regressor" additionally make `t.fit()` log
#                   `val_accuracy` / `val_r2`. None skips the tag.
#   weights_path  — where `t.save(model)` writes the pickle.
#                   Default "src/weights.pkl" matches `register(artifact_dirs="src")`.
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
)
from nubison_model import train

MODEL_ID = "distilbert-base-uncased"


def _as_hf_dataset(X, y, tokenizer):
    df = X.assign(labels=y.values.ravel())
    ds = Dataset.from_pandas(df, preserve_index=False)
    return ds.map(
        lambda r: tokenizer(r["text"], padding="max_length", truncation=True, max_length=16),
        batched=True,
    )


with train(datasets=datasets, target=["target"], model_type="classifier") as t:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

    train_ds = _as_hf_dataset(t.X_train, t.y_train, tokenizer)
    val_ds = _as_hf_dataset(t.X_val, t.y_val, tokenizer)

    args = TrainingArguments(
        output_dir="/tmp/hf_trainer",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        eval_strategy="epoch",
        logging_steps=1,
        report_to=["mlflow"],
        disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
    trainer.train()

    t.save(model)

print(f"run_id: {t.run_id}")
